# [Baseline] RandomForest — 풍력발전량 예측

기상 예보 데이터(LDAPS/GFS)로 KPX 발전그룹 3곳의 시간대별 **풍력발전량(kWh)** 을 예측하는 회귀 베이스라인입니다.

- **입력**: 예측 시각별 LDAPS/GFS 기상 예보 격자 평균값 + 시간·요일·계절성 캘린더 변수
- **출력**: `kpx_group_1/2/3` 세 발전그룹의 시간대별 예측 발전량(kWh)
- **모델**: 그룹별 `RandomForestRegressor` (Label 제공 기간이 그룹마다 다르므로 각각 따로 학습)

이 대회는 예측 결과 CSV(`baseline_submit.csv`)를 제출하는 방식이라, 데이터 로드부터 제출 파일 생성까지
이 노트북 하나로 처리합니다.


## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 회귀 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer


## 2. 데이터 로드

발전량 Label(`train_labels.csv`), 제출 양식(`sample_submission.csv`), LDAPS/GFS 기상 예보 데이터를 불러오고,
날짜 컬럼을 datetime으로 변환합니다. `CAPACITY_KWH` 는 그룹별 설비 용량으로, 이후 예측값을 클리핑하는 데
사용합니다.


In [ ]:
DATA_DIR = Path(".")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("train_labels:", train_labels.shape)
print("sample_submission:", sample_submission.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, "gfs_test:", gfs_test.shape)


## 3. Feature 생성

LDAPS/GFS는 하나의 예측 시각에 여러 격자가 존재하므로, `forecast_kst_dtm`별로 수치형 기상변수의 평균값을 계산합니다.

추가로 월, 일, 시간, 요일, 주말 여부, 시간/월의 주기성을 나타내는 sin-cos feature를 생성합니다.


In [ ]:
def aggregate_weather(df, prefix):
    df = df.copy()
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    agg = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}_mean" for c in agg.columns]
    return agg.reset_index()


def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out


train_weather = aggregate_weather(ldaps_train, "ldaps").merge(
    aggregate_weather(gfs_train, "gfs"), on="forecast_kst_dtm", how="inner"
)
test_weather = aggregate_weather(ldaps_test, "ldaps").merge(
    aggregate_weather(gfs_test, "gfs"), on="forecast_kst_dtm", how="inner"
)

train_base = train_labels.rename(columns={"kst_dtm": "forecast_kst_dtm"})
train_df = train_base.merge(train_weather, on="forecast_kst_dtm", how="left")
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
    test_weather, on="forecast_kst_dtm", how="left"
)

X_train = pd.concat(
    [calendar_features(train_df["forecast_kst_dtm"]), train_df.drop(columns=["forecast_kst_dtm", *TARGET_COLS])],
    axis=1,
)
X_test = pd.concat(
    [calendar_features(test_df["forecast_kst_dtm"]), test_df.drop(columns=["forecast_id", "forecast_kst_dtm"])],
    axis=1,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## 4. 모델 학습

KPX 그룹별로 Label 제공 기간이 다르므로, 각 그룹의 Label이 존재하는 행만 사용해 RandomForest 모델을 따로 학습합니다.

RandomForest는 여러 결정트리를 앙상블하는 배깅(Bagging) 모델로, 각 트리를 부트스트랩 샘플과 랜덤 피처
subset으로 학습한 뒤 예측을 평균 냅니다.


In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

predictions = pd.DataFrame(index=sample_submission.index)

for target in TARGET_COLS:
    train_mask = train_df[target].notna()
    y_train = train_df.loc[train_mask, target]

    model = RandomForestRegressor(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=8,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_imp.loc[train_mask], y_train)

    pred = model.predict(X_test_imp)
    pred = np.clip(pred, 0, CAPACITY_KWH[target])
    predictions[target] = pred

    print(target, "train rows:", int(train_mask.sum()))


## 5. 테스트 데이터 예측 생성

학습한 그룹별 모델로 평가 기간 전체의 발전량을 예측합니다. 예측값은 0 이상, 설비 용량 이하로 클리핑하여
물리적으로 불가능한 값을 방지합니다.


In [ ]:
submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
for target in TARGET_COLS:
    submission[target] = predictions[target]

submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

print(submission.head())
print(submission.shape)


## 6. baseline_submit.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다. 이 대회는 결과 CSV를 직접 제출하는
방식이므로, 이 파일을 그대로 제출하면 됩니다.


In [ ]:
output_path = DATA_DIR / "baseline_submit.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")
